# Diagnostik download HuggingFace — HANYA MEMBACA

Notebook ini **tidak menghapus apa pun**. Ia cuma mencetak keadaan, supaya kita lihat datanya sebelum memutuskan perbaikan.

## Cara pakai

Kernel yang diperiksa harus kernel yang **benar-benar gagal** — bukan runtime segar. Jalankan `04_extract_all.ipynb` sampai error 403 muncul, lalu jalankan dua cell di bawah **di kernel yang sama**.

## Kenapa versi 1 dari notebook ini error

Cell A yang lama memanggil `from huggingface_hub.utils import is_xet_available` — dan itu meledak `ImportError` di Colab. Penyebabnya: `huggingface_hub` di Colab **bukan** versi 1.23.0 yang ada di laptop, dan API-nya berbeda.

Artinya seluruh penalaran kita soal xet sejauh ini berdiri di atas versi yang salah. Jadi hal **pertama** yang harus notebook ini cetak adalah versi `huggingface_hub` yang sebenarnya di Colab — baru setelah itu kita boleh menyimpulkan apa pun.

Cell A sekarang defensif: tidak mengasumsikan API versi mana pun, dan tidak akan mati kalau sebuah fungsi tidak ada.

In [ ]:
# === BAGIAN A: jalur download + status auth ===
# Defensif: JANGAN asumsikan API versi mana pun. Cetak versinya dulu.
import importlib
import importlib.util
import os
import sys
from pathlib import Path

# Dicek SEBELUM kita sendiri meng-import huggingface_hub di bawah.
print("huggingface_hub sudah ter-import :", "huggingface_hub" in sys.modules)
print("HF_HUB_DISABLE_XET (os.environ)  :",
      repr(os.environ.get("HF_HUB_DISABLE_XET", "<tidak di-set>")))

import huggingface_hub
from huggingface_hub import constants

print("huggingface_hub.__version__      :", huggingface_hub.__version__)   # <-- ANGKA YANG KUBUTUHKAN
print("lokasi                           :", huggingface_hub.__file__)

print("transformers                     :", end=" ")
try:
    import transformers
    print(transformers.__version__)
except ImportError:
    print("belum ter-import")

hf_xet_installed = importlib.util.find_spec("hf_xet") is not None
disable_const = getattr(constants, "HF_HUB_DISABLE_XET", "<konstanta tidak ada di versi ini>")

print("paket hf_xet terpasang           :", hf_xet_installed)
print("constants.HF_HUB_DISABLE_XET     :", disable_const)

# --- TOKEN: cek os.environ SAJA TIDAK CUKUP ---
# login() menulis token ke FILE (~/.cache/huggingface/token), BUKAN ke environment.
# Jadi os.environ bisa kosong padahal kamu login sah. get_token() membaca dua-duanya:
# env var DAN file.
print()
print("HF_TOKEN di os.environ           :",
      "ada" if (os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"))
      else "tidak ada")

_get_token = getattr(huggingface_hub, "get_token", None)
if _get_token is None:                                   # versi lama
    _get_token = huggingface_hub.HfFolder.get_token
tok = _get_token()
print("get_token() (env ATAU file)      :",
      f"ADA ({tok[:4]}...{tok[-4:]}, {len(tok)} char)" if tok else "TIDAK ADA")

tok_file = Path(getattr(constants, "HF_TOKEN_PATH", "~/.cache/huggingface/token")).expanduser()
print("file token                       :", f"{tok_file} (ada: {tok_file.exists()})")

# Token "ada" belum berarti token itu SAH. Buktikan ke server, jangan diasumsikan.
try:
    me = huggingface_hub.whoami(token=tok)
    print("whoami()                         :", me.get("name"), f"({me.get('type')})")
except Exception as e:
    print("whoami() GAGAL                   :", type(e).__name__, str(e)[:150])

# Header Authorization yang BENAR-BENAR ikut terkirim saat download.
try:
    from huggingface_hub.utils import build_hf_headers
    hdrs = build_hf_headers(token=tok)
    print("header Authorization terkirim    :",
          "YA" if "authorization" in {k.lower() for k in hdrs} else "TIDAK")
except Exception as e:
    print("build_hf_headers tidak tersedia  :", type(e).__name__)

# --- JALUR DOWNLOAD ---
# is_xet_available pindah lokasi antar versi -- coba semua, jangan asumsi satu pun.
fn = None
for modpath in ("huggingface_hub.utils", "huggingface_hub.utils._runtime", "huggingface_hub"):
    try:
        fn = getattr(importlib.import_module(modpath), "is_xet_available", None)
    except ImportError:
        continue
    if fn:
        print("\nis_xet_available ditemukan di    :", modpath)
        break

print()
if fn:
    print(">>> is_xet_available() =", fn())
else:
    print(">>> is_xet_available() TIDAK ADA di versi ini.")
    print("    Perkiraan jalur xet aktif =", hf_xet_installed and not disable_const)

In [ ]:
# === BAGIAN B: isi cache HF (lihat saja, jangan hapus) ===
from pathlib import Path

from huggingface_hub import constants


def _human(nbytes):
    x = float(nbytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if x < 1024 or unit == "TB":
            return f"{x:7.1f} {unit}"
        x /= 1024


def _tree_size(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())


# Pakai konstanta HF, bukan hardcode ~/.cache/huggingface -- HF_HOME bisa dipindah.
# getattr: nama konstanta pun bisa beda antar versi, jangan bikin cell ini mati lagi.
hub_cache = Path(getattr(constants, "HF_HUB_CACHE", "~/.cache/huggingface/hub")).expanduser()
xet_cache = Path(getattr(constants, "HF_XET_CACHE", "~/.cache/huggingface/xet")).expanduser()

print("HF_HOME      :", getattr(constants, "HF_HOME", "<tidak ada>"))
print(f"HF_HUB_CACHE : {hub_cache}  (ada: {hub_cache.exists()})")
print(f"HF_XET_CACHE : {xet_cache}  (ada: {xet_cache.exists()})")

print("\n--- repo di hub cache ---")
repos = sorted(d for d in hub_cache.iterdir()
               if d.is_dir() and d.name.startswith("models--")) if hub_cache.exists() else []
if not repos:
    print("  (kosong -- tidak ada model ter-cache)")
for d in repos:
    print(f"  {_human(_tree_size(d))}  {d.name}")

print("\n--- file .incomplete (sisa download yang mati di tengah) ---")
incomplete = sorted(hub_cache.rglob("*.incomplete")) if hub_cache.exists() else []
if not incomplete:
    print("  TIDAK ADA. Hipotesis 'cache parsial/korup' tidak didukung data.")
for f in incomplete:
    print(f"  {_human(f.stat().st_size)}  {f.relative_to(hub_cache)}")

print("\n--- xet cache ---")
if xet_cache.exists():
    n_files = sum(1 for f in xet_cache.rglob("*") if f.is_file())
    print(f"  {_human(_tree_size(xet_cache))} dalam {n_files} file")
    for d in sorted(p for p in xet_cache.iterdir() if p.is_dir()):
        print(f"    {_human(_tree_size(d))}  {d.name}/")
else:
    print("  (tidak ada -- xet belum pernah dipakai di runtime ini)")

print("\nSelesai. TIDAK ADA yang dihapus.")

## Yang perlu dilaporkan balik

Tempel **seluruh output** kedua cell. Yang paling menentukan, urut prioritas:

1. **`huggingface_hub.__version__`** — tanpa ini semua analisis kode cuma tebakan.
2. `paket hf_xet terpasang` → `True`/`False`
3. `is_xet_available()` (atau perkiraannya kalau fungsinya tidak ada)
4. Ada tidaknya file `.incomplete`

Jangan hapus cache dulu. Kalau `.incomplete` memang ada **dan** jalur xet ternyata mati tapi 403 tetap muncul, berarti hipotesis cache-mu yang benar dan diagnosa xet-ku meleset — dan itu harus ketahuan dari data, bukan dari asumsi siapa pun.